Processing of the regional marine currents data to get regional averages 

In [14]:
import pandas as pd
import numpy as np

# Read the CSV file
df = pd.read_csv('Velocities_all.csv')

# Density mapping
density_map = {
    'EPS': 1,
    'HDPE': 4,
    'LDPE': 3,
    'PET': 6,
    'PP': 2,
    'PS': 5
}

# Add density code column
df['Density_code'] = df['Polymer_all'].map(density_map)

# Define regions and their countries with weights
regions = {
    'Northern regions': {
        'KaraSea': 7.96E+02
    },
    'Central Asia': {
        'SeaOkhotsk': 3.83E+04
    },
    'Southeast Asia': {
        'EC-Japan': 1.47E+06,
        'BayBengal': 7.28E+04
    },
    'Oceania': {
        'Oceania': 1.53E+03
    },
    'Europe': {
        'Greece': 7.32E+01,
        'NorthSea': 4.73E+02
    },
    'Africa & Middle east': {
        'Nile': 8.48E+04,
        'GG_Ext': 3.52E+04,
        'Congo': 1.34E+04
    },
    'North America': {
        'NorthAmerica': 5.72E+02
    },
    'Latin America': {
        'SouthAmerica': 3.73E+03
    }
}

# Columns that need weighted averaging (only v_ and k_ columns)
weighted_columns = [
    'v_Surf2Col_cont_median_all',
    'v_Col2Sed_cont_median_all',
    'k_Surf2glo_median_all',
    'k_Col2glo_median_all',
    'v_Surf2Col_glo_median_all',
    'v_Col2Sed_glo_median_all'
]

# Create a list to store results
results = []

# Group by Polymer, Density, Shape, Size, Diameter
grouping_cols = ['Polymer_all', 'Density_all', 'Shape_all', 'Size_all', 'Diameter_all']

for group_keys, group_df in df.groupby(grouping_cols):
    # Process each region
    for region_name, country_weights in regions.items():
        row_data = {
            'Region': region_name,
            'Polymer_all': group_keys[0] if isinstance(group_keys, tuple) else group_keys,
            'Density_all': group_keys[1] if isinstance(group_keys, tuple) else None,
            'Shape_all': group_keys[2] if isinstance(group_keys, tuple) else None,
            'Size_all': group_keys[3] if isinstance(group_keys, tuple) else None,
            'Diameter_all': group_keys[4] if isinstance(group_keys, tuple) else None,
            'Density_code': group_df.iloc[0]['Density_code']
        }
        
        # Track which countries had data for this combination
        countries_used = []
        
        # Calculate weighted averages for each column
        for col in weighted_columns:
            total_weight = 0
            weighted_sum = 0
            
            for country, weight in country_weights.items():
                # Find matching country in this group
                country_data = group_df[group_df['Country_all'] == country]
                
                if not country_data.empty:
                    value = country_data[col].iloc[0]
                    # Only include if value is not NaN
                    if not pd.isna(value):
                        weighted_sum += value * weight
                        total_weight += weight
                        if country not in countries_used:
                            countries_used.append(country)
            
            # Calculate weighted average (handle case where all values were NaN)
            if total_weight > 0:
                row_data[col] = weighted_sum / total_weight
            else:
                row_data[col] = 0
        
        # Add the list of countries used
        row_data['Country_all'] = ';'.join(countries_used) if countries_used else ''
        
        results.append(row_data)

# Create result dataframe
result_df = pd.DataFrame(results)

# Define the exact column order as requested
column_order = [
    'Region',
    'Country_all',
    'Polymer_all',
    'Density_code',
    'Density_all',
    'Shape_all',
    'Size_all',
    'Diameter_all',
    'v_Surf2Col_cont_median_all',
    'v_Col2Sed_cont_median_all',
    'k_Surf2glo_median_all',
    'k_Col2glo_median_all',
    'v_Surf2Col_glo_median_all',
    'v_Col2Sed_glo_median_all'
]

# Ensure all columns exist (fill missing with 0)
for col in column_order:
    if col not in result_df.columns:
        result_df[col] = 0

result_df = result_df[column_order]

# Replace any remaining NaN with 0
result_df = result_df.fillna(0)

# Sort by Region, Polymer, Shape, Size
# Create custom sorting order for regions to match the specified order
region_order = [
    'Northern regions',
    'Central Asia',
    'Southeast Asia',
    'Oceania',
    'Europe',
    'Africa & Middle east',
    'North America',
    'Latin America'
]

# Convert Region to categorical with custom order
result_df['Region'] = pd.Categorical(result_df['Region'], categories=region_order, ordered=True)

# Sort the dataframe
result_df = result_df.sort_values(
    by=['Region', 'Polymer_all', 'Shape_all', 'Size_all'],
    ascending=[True, True, True, True]
)

# Reset index after sorting
result_df = result_df.reset_index(drop=True)

# Save to Excel
result_df.to_excel('Velocities_all_weighted.xlsx', index=False, sheet_name='Weighted_Velocities')

print(f"Processing complete!")
print(f"Created {len(results)} rows (combinations of polymer properties × regions)")
print(f"Output saved to: Velocities_all_weighted.xlsx")

# Display first few rows as preview
print("\nPreview of first 10 rows:")
print(result_df.head(10).to_string())

# Show unique regions to verify Africa & Middle east spelling
print("\nRegions in output:")
print(result_df['Region'].unique())

Processing complete!
Created 480 rows (combinations of polymer properties × regions)
Output saved to: Velocities_all_weighted.xlsx

Preview of first 10 rows:
             Region Country_all Polymer_all  Density_code  Density_all Shape_all  Size_all  Diameter_all  v_Surf2Col_cont_median_all  v_Col2Sed_cont_median_all  k_Surf2glo_median_all  k_Col2glo_median_all  v_Surf2Col_glo_median_all  v_Col2Sed_glo_median_all
0  Northern regions     KaraSea         EPS             1         0.03     fiber  0.000050      0.000001                    0.052349                   0.984557                    0.0                   0.0                        0.0                       0.0
1  Northern regions     KaraSea         EPS             1         0.03     fiber  0.000100      0.000010                    0.245175                   3.427145                    0.0                   0.0                        0.0                       0.0
2  Northern regions     KaraSea         EPS             1         0.